# 09_Feature_Recalculation (Final)
Final feature recalculation notebook for HSEP.

In [2]:

import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler


In [3]:

tech = pd.read_csv('../Generated Datasets/technical_skill_master.csv')
history = pd.read_csv('../Generated Datasets/skill_demand_history_clean.csv')
global_adoption = pd.read_csv('../Generated Datasets/global_adoption_stackoverflow.csv')

postings = pd.read_csv('../Raw Data/postings.csv')


In [4]:

features = tech.copy()

features['linkedin_demand'] = (
    features['linkedin_frequency'] /
    features['linkedin_frequency'].max()
)


In [5]:
salary_col = 'normalized_salary'

postings[salary_col] = pd.to_numeric(
    postings[salary_col],
    errors='coerce'
)

text_cols = [
    c for c in postings.columns
    if postings[c].dtype == 'object'
]

preferred_cols = [
    c for c in postings.columns
    if 'description' in c.lower()
]

text_col = preferred_cols[0] if preferred_cols else text_cols[0]

salary_scores = []

for skill in features['skill']:

    mask = postings[text_col].astype(str).str.contains(
        rf'\b{re.escape(skill)}\b',
        case=False,
        regex=True,
        na=False
    )

    subset = postings.loc[mask]

    val = subset[salary_col].median() if len(subset) else np.nan

    salary_scores.append(val)

features['salary_premium_raw'] = salary_scores

subcat_medians = (
    features.groupby('sub_category')['salary_premium_raw']
    .median()
)

features['salary_premium_raw'] = features.apply(
    lambda x: subcat_medians.get(x['sub_category'])
    if pd.isna(x['salary_premium_raw'])
    else x['salary_premium_raw'],
    axis=1
)

features['salary_premium_raw'] = (
    features['salary_premium_raw']
    .fillna(features['salary_premium_raw'].median())
)

features['salary_premium'] = MinMaxScaler().fit_transform(
    features[['salary_premium_raw']]
)


In [6]:

latest_year = history['year'].max()

latest = (
    history[history['year']==latest_year]
    [['skill','adoption_rate']]
    .rename(columns={'adoption_rate':'current_usage_raw'})
)

features = features.merge(
    latest,
    on='skill',
    how='left'
)


In [7]:

survey = pd.read_csv('../Raw Data/StackOverflow/survey_2025.csv')

want_cols = [
    c for c in survey.columns
    if 'WantToWorkWith' in c
]

interest_counts = {}

for col in want_cols:

    for row in survey[col].dropna().astype(str):

        for item in row.split(';'):

            item = item.strip().lower()

            interest_counts[item] = (
                interest_counts.get(item,0)+1
            )

features['future_interest_raw'] = (
    features['skill']
    .str.lower()
    .map(interest_counts)
    .fillna(0)
)


C:\Users\Saanvi\AppData\Local\Temp\ipykernel_18692\1679708315.py:1: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  survey = pd.read_csv('../Raw Data/StackOverflow/survey_2025.csv')


In [8]:

score_col = [
    c for c in global_adoption.columns
    if c.lower() != 'skill'
][0]

global_adoption = global_adoption.rename(
    columns={score_col:'global_adoption_score'}
)

features = features.merge(
    global_adoption[['skill','global_adoption_score']],
    on='skill',
    how='left'
)

features['global_adoption_score'] = (
    features['global_adoption_score']
    .fillna(features['global_adoption_score'].median())
)


In [9]:

growth_map = {}

for skill, grp in history.groupby('skill'):

    grp = grp.sort_values('year')

    if len(grp) < 2:
        continue

    years = grp['year'].values
    adoption = grp['adoption_rate'].values

    slope = np.polyfit(years, adoption, 1)[0]

    recent_growth = 0

    if len(grp) >= 3:

        recent = grp.tail(3)

        first = recent['adoption_rate'].iloc[0]
        last = recent['adoption_rate'].iloc[-1]

        if first != 0:
            recent_growth = (last-first)/first

    growth_map[skill] = (
        0.7*slope +
        0.3*recent_growth
    )

features['growth_rate_raw'] = (
    features['skill']
    .map(growth_map)
)

subcat_growth = (
    features.groupby('sub_category')['growth_rate_raw']
    .median()
)

features['growth_rate_raw'] = features.apply(
    lambda x: subcat_growth.get(x['sub_category'])
    if pd.isna(x['growth_rate_raw'])
    else x['growth_rate_raw'],
    axis=1
)


In [10]:

for col in [
    'current_usage_raw',
    'future_interest_raw',
    'growth_rate_raw'
]:
    features[col] = features[col].fillna(0)

features['current_usage'] = MinMaxScaler().fit_transform(
    features[['current_usage_raw']]
)

features['future_interest'] = MinMaxScaler().fit_transform(
    features[['future_interest_raw']]
)

features['growth_rate'] = MinMaxScaler().fit_transform(
    features[['growth_rate_raw']]
)


In [11]:

final_df = features[[
    'skill',
    'sub_category',
    'linkedin_demand',
    'salary_premium',
    'current_usage',
    'future_interest',
    'global_adoption_score',
    'growth_rate'
]].copy()

final_df['history_found'] = (
    final_df['skill']
    .isin(history['skill'])
    .astype(int)
)

print(final_df.shape)
final_df.head()


(114, 10)


,skill,sub_category,linkedin_demand,salary_premium,current_usage,future_interest,global_adoption_score,global_adoption_score,growth_rate,history_found
0,data analysis,Data Analytics,1.000000,0.326209,0.000000,0.000000,160.0,160.000000,0.566533,0
1,excel,Data Analytics,0.506295,0.067637,0.000000,0.000000,160.0,160.000000,0.566533,0
2,quality assurance,Software Engineering,0.442621,0.175857,0.000000,0.000000,160.0,160.000000,0.566533,0
3,microsoft excel,Data Analytics,0.314443,0.067637,0.000000,0.000000,160.0,160.000000,0.566533,0
4,python,Programming,0.307062,0.631282,0.876458,0.701599,173.0,0.936412,0.255955,1


In [13]:
ai_reference = final_df[
    final_df['skill'].isin([
        'artificial intelligence',
        'natural language processing'
    ])
]

base = ai_reference.mean(numeric_only=True)

new_skills = pd.DataFrame({
    'skill': ['llm', 'langchain', 'rag'],
    'sub_category': ['AI_ML', 'AI_ML', 'AI_ML'],
    'linkedin_demand': [base['linkedin_demand']] * 3,
    'salary_premium': [base['salary_premium']] * 3,
    'current_usage': [base['current_usage']] * 3,
    'future_interest': [min(1.0, base['future_interest'] * 1.15)] * 3,
    'global_adoption_score': [base['global_adoption_score']] * 3,
    'growth_rate': [min(1.0, base['growth_rate'] * 1.20)] * 3,
    'history_found': [0, 0, 0]
})

print(final_df.columns.tolist())

final_df = final_df.loc[:, ~final_df.columns.duplicated()]

if 'global_adoption_score.1' in final_df.columns:
    final_df = final_df.drop(columns=['global_adoption_score.1'])

print(final_df.columns.tolist())
print(final_df.shape)

print("Updated Shape:", final_df.shape)
print(final_df.tail(3))

final_df.to_csv(
    '../Generated Datasets/expanded_skill_master.csv',
    index=False
)  

['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'global_adoption_score', 'global_adoption_score', 'growth_rate', 'history_found']
['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'global_adoption_score', 'growth_rate', 'history_found']
(114, 9)
Updated Shape: (114, 9)
                   skill          sub_category  linkedin_demand  \
111  performance testing  Software Engineering         0.012457   
112  penetration testing         Cybersecurity         0.012347   
113      nosql databases              Database         0.012310   

     salary_premium  current_usage  future_interest  global_adoption_score  \
111        0.477684            0.0              0.0                  160.0   
112        0.631282            0.0              0.0                  160.0   
113        0.674119            0.0              0.0                  160.0   

     growth_rate  history_found  
111     0.5665

In [14]:

final_df.to_csv(
    '../Generated Datasets/expanded_skill_master.csv',
    index=False
)

print("Saved successfully")
print(final_df.shape)
print(final_df.columns.tolist())
print(final_df.isna().sum())


Saved successfully
(114, 9)
['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'global_adoption_score', 'growth_rate', 'history_found']
skill                    0
sub_category             0
linkedin_demand          0
salary_premium           0
current_usage            0
future_interest          0
global_adoption_score    0
growth_rate              0
history_found            0
dtype: int64


In [15]:
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

In [16]:
print(final_df.shape)
print(final_df.columns.tolist())

(114, 9)
['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'global_adoption_score', 'growth_rate', 'history_found']


In [17]:

final_df.to_csv(
    '../Generated Datasets/expanded_skill_master.csv',
    index=False
)

print("Saved successfully")
print(final_df.shape)
print(final_df.columns.tolist())
print(final_df.isna().sum())


Saved successfully
(114, 9)
['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'global_adoption_score', 'growth_rate', 'history_found']
skill                    0
sub_category             0
linkedin_demand          0
salary_premium           0
current_usage            0
future_interest          0
global_adoption_score    0
growth_rate              0
history_found            0
dtype: int64
